In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [3]:
df = sns.load_dataset('iris')

In [4]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
df['species'].value_counts()

,count
species,
setosa,50
versicolor,50
virginica,50


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X = df.drop('species', axis=1)
y = df['species']

In [127]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [128]:
from sklearn.neighbors import KNeighborsClassifier

In [129]:
model_knn = KNeighborsClassifier(n_neighbors=1)

In [130]:
model_knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=1)

In [131]:
round(model_knn.score(X_train, y_train)*100,2)

100.0

In [132]:
from sklearn.svm import SVC

In [133]:
model_SVM = SVC(gamma='auto')

In [134]:
model_SVM.fit(X_train, y_train)

SVC(gamma='auto')

In [135]:
round(model_knn.score(X_train, y_train)*100,2)

100.0

In [136]:
# Now lets use the GridSearchCV for not manually find the best hyperparamter tuning
# Our model is overfitting right now but cross validation help us gets rid of overfitting
# so after applying the gridsearchcv it will be great

In [137]:
from sklearn.model_selection import GridSearchCV, cross_val_score

In [138]:
classifier_svm = GridSearchCV(model_SVM, {
    'C': [1,10,20,30],
    'kernel': ['rbf', 'linear'],
}, cv=5, return_train_score=False)

In [139]:
classifier_svm.fit(X,y)

GridSearchCV(cv=5, estimator=SVC(gamma='auto'),
             param_grid={'C': [1, 10, 20, 30], 'kernel': ['rbf', 'linear']})

In [140]:
classifier_svm.cv_results_

{'mean_fit_time': array([0.00611944, 0.0044271 , 0.00435758, 0.00411339, 0.00449562,
        0.00388746, 0.00419888, 0.00439301]),
 'std_fit_time': array([0.00310258, 0.0007329 , 0.00014645, 0.00029676, 0.00036085,
        0.00015961, 0.00013923, 0.00052195]),
 'mean_score_time': array([0.00417738, 0.00342102, 0.00333142, 0.00358667, 0.00332885,
        0.00302806, 0.00311513, 0.00339937]),
 'std_score_time': array([1.17772663e-03, 4.05765795e-04, 1.90361193e-04, 6.74103016e-04,
        3.32184985e-04, 8.73703873e-05, 6.89712272e-05, 5.25140418e-04]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20, 30, 30],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear',
                    'rbf', 'linear'],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [

In [141]:
results_svm = pd.DataFrame(classifier_svm.cv_results_)

In [142]:
results_svm[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667
6,30,rbf,0.960000
7,30,linear,0.960000


In [143]:
classifier_knn = GridSearchCV(model_knn, {
    "n_neighbors": [1,3,5,7,9,11,13,15,17,19,21,23,25],
    "weights": ['uniform', 'distance'],
    "metric": ["manhattan", "euclidean"],
}, cv=5, return_train_score=False)

In [144]:
classifier_knn.fit(X, y)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(n_neighbors=1),
             param_grid={'metric': ['manhattan', 'euclidean'],
                         'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21,
                                         23, 25],
                         'weights': ['uniform', 'distance']})

In [145]:
results_knn = pd.DataFrame(classifier_knn.cv_results_)

In [146]:
results_svm[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667
6,30,rbf,0.960000
7,30,linear,0.960000


In [147]:
# Evaluating the tuned KNN model on completely unseen test data.

In [148]:
model_knn = classifier_knn.best_estimator_ # storing the best hyper parameter we get from the
# grid search cv

In [149]:
y_pred_knn = model_knn.predict(X_test) # storing prediction

In [150]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [151]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn) * 100:.2f}%")
print("Precision:", precision_score(y_test, y_pred_knn, average="weighted"))
print("Recall:", recall_score(y_test, y_pred_knn, average="weighted"))
print("F1:", f1_score(y_test, y_pred_knn, average="weighted"))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy: 100.00%
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion Matrix:
[[29  0  0]
 [ 0 23  0]
 [ 0  0 23]]
